# AutoGen: Basic to Advanced — Classroom Annotated Notebook

This notebook introduces **AutoGen agents step by step**, beginning with a simple conversation between a user and an AI assistant and gradually moving toward:

- Human-in-the-loop conversations
- Context-aware troubleshooting
- Dynamically configured agents
- Multi-agent group chats
- AutoGen and LangGraph integration
- Tool-enabled agents
- OCR-based document analysis

## Learning objective

By the end of the notebook, students should understand:

1. What an AutoGen agent is
2. The difference between `AssistantAgent` and `UserProxyAgent`
3. How agents exchange messages
4. How system messages define agent behaviour
5. How multiple specialised agents collaborate
6. How LangGraph adds explicit workflow control
7. How agents can call external tools
8. How a multi-agent system can analyse a document

> **Classroom note:** Run the notebook section by section. Several sections ask for API keys and start an interactive chat, so they should not all be executed at once.


## 1. Install AutoGen

The first cell installs the packages required for the examples.

- `pyautogen` provides the AutoGen agent framework.
- `openai` provides an OpenAI-compatible client interface.
- `autogen` is included here for compatibility with the notebook imports.

In Colab, installation cells generally need to be run only once per runtime.


In [1]:
!pip install pyautogen openai autogen


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 6.1 MB/s eta 0:00:00


## 2. Create the User Proxy Agent

AutoGen applications are built using agents that send messages to one another.

This first agent is a `UserProxyAgent`. It represents the human user inside the AutoGen conversation.

### Important parameters

- `name="customer"` gives the agent an identity in the conversation.
- `human_input_mode="ALWAYS"` means the framework asks the human for input whenever this agent gets a turn.
- `code_execution_config={"use_docker": False}` disables Docker-based code execution.
- `max_consecutive_auto_reply=5` limits automatic replies and helps prevent an endless conversation.

The API key is collected using `getpass`, so it is not displayed openly in the notebook output.


In [2]:
# -----------------------------------------
# Imports
# -----------------------------------------
import os
import autogen
import warnings
warnings.filterwarnings("ignore")

# -----------------------------------------
# Load API key
# -----------------------------------------
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API Key: ")

# -----------------------------------------
# Step 1: Define Customer Agent
# -----------------------------------------
customer_agent = autogen.UserProxyAgent(
    name="customer",
    human_input_mode="ALWAYS"
)



Enter your OpenRouter API Key: ··········


## 3. Create the Assistant Agent and Start a Chat

`AssistantAgent` represents an LLM-powered AI participant.

The `llm_config` tells AutoGen:

- Which model to use
- Which API key to send
- Which OpenAI-compatible endpoint to call
- What temperature to use

The `system_message` defines the role and behaviour of the support agent.

```text
Customer/UserProxyAgent
          │
          │ sends the initial message
          ▼
     Support AssistantAgent
          │
          │ generates a response
          ▼
       Customer
```

`initiate_chat()` begins the message exchange. The first argument is the receiving agent, and `message` is the initial user request.


In [3]:
# -----------------------------------------
# Step 2: Define Support Agent (Using OpenRouter)
# -----------------------------------------
support_agent = autogen.AssistantAgent(
    name="support_agent",
    llm_config={
        "config_list": [
            {
                "model": "openai/gpt-4o-mini",
                "api_key": os.getenv("OPENROUTER_API_KEY"),
                "base_url": "https://openrouter.ai/api/v1",
                "price": [0.000, 0.000],
            }
        ],
        "temperature": 0.7,
    },
    system_message=(
        "You are a helpful AI support agent. "
        "Answer customer queries clearly and professionally."
    ),
)

# -----------------------------------------
# Step 3: Start Conversation
# -----------------------------------------
ip = input('Enter input:')
customer_agent.initiate_chat(
    support_agent,
    message=ip
)

Enter input:My excel is not working
customer (to support_agent):

My excel is not working

--------------------------------------------------------------------------------
support_agent (to customer):

I'm sorry to hear that you're having trouble with Excel. Can you please provide more details about the issue you're experiencing? For example, are you receiving any error messages, or is Excel not opening at all? Any additional information will help me assist you better.

--------------------------------------------------------------------------------
Replying as customer. Provide feedback to support_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: Not opening at all
customer (to support_agent):

Not opening at all

--------------------------------------------------------------------------------
support_agent (to customer):

Thank you for the information. If Excel is not opening at all, here are some steps you can try to resolve the issue:

1. **Rest

ChatResult(chat_id=239497801417372179191426425609094196991, chat_history=[{'content': 'My excel is not working', 'role': 'assistant', 'name': 'customer'}, {'content': "I'm sorry to hear that you're having trouble with Excel. Can you please provide more details about the issue you're experiencing? For example, are you receiving any error messages, or is Excel not opening at all? Any additional information will help me assist you better.", 'role': 'user', 'name': 'support_agent'}, {'content': 'Not opening at all', 'role': 'assistant', 'name': 'customer'}, {'content': 'Thank you for the information. If Excel is not opening at all, here are some steps you can try to resolve the issue:\n\n1. **Restart Your Computer**: Sometimes a simple restart can resolve software issues.\n\n2. **Check for Updates**: Ensure that your version of Microsoft Office is up to date. You can check for updates by opening another Office application (like Word) and going to `File` > `Account` > `Update Options`.\n\n3

# Part 2: Context-Aware IT Helpdesk Agent

The next examples use an IT support scenario to demonstrate longer conversations and human feedback.

AutoGen maintains the messages exchanged during a chat session. Because previous messages remain part of the conversation, the assistant can use earlier troubleshooting steps and user responses as context.

> This is conversation history within the current chat. It is not permanent long-term memory unless the history is stored externally and loaded again later.


## 4. Human-in-the-Loop Troubleshooting

This example creates:

- An `AssistantAgent` that gives IT troubleshooting steps
- A `UserProxyAgent` that collects human responses

The assistant's system message asks it to provide a troubleshooting step and then ask whether the step solved the problem.

Because `human_input_mode="ALWAYS"`, the user remains involved throughout the conversation.

### Message flow

```text
User describes issue
        ↓
IT agent suggests a step
        ↓
IT agent asks: Did this solve the issue?
        ↓
Human answers yes or no
        ↓
Conversation continues using previous context
```


In [4]:


# ==========================================
# Setup API Key
# ==========================================
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API Key: ")

# ==========================================
# Imports
# ==========================================
import autogen

# ==========================================
# LLM Config for OpenRouter
# ==========================================
llm_config = {
    "config_list": [
        {
            "model": "openai/gpt-4o-mini",
            "api_key": os.getenv("OPENROUTER_API_KEY"),
            "base_url": "https://openrouter.ai/api/v1",
            "price": [0, 0],
        }
    ],
    "temperature": 0.4,
}

# ==========================================
# Define IT Support Agent
# ==========================================
it_support_agent = autogen.AssistantAgent(
    name="IT_Support_Bot",
    llm_config=llm_config,
    system_message="""
You are a helpful IT support assistant.
Provide troubleshooting steps clearly and professionally.
Ask after each step: 'Did this solve your issue? (yes/no)'
""",
)

# ==========================================
# Define User Agent
# ==========================================
user_agent = autogen.UserProxyAgent(
    name="User",
    human_input_mode="ALWAYS"
)

# ==========================================
# Start Chat
# ==========================================
issue = input("Describe your IT issue: ")

user_agent.initiate_chat(
    it_support_agent,
    message=issue
)


Enter your OpenRouter API Key: ··········
Describe your IT issue: Excel is not working
User (to IT_Support_Bot):

Excel is not working

--------------------------------------------------------------------------------
IT_Support_Bot (to User):

I'm here to help you troubleshoot the issue with Excel. Let's go through some steps to identify and resolve the problem.

1. **Restart Excel**: Close Excel completely and then reopen it. Sometimes, a simple restart can resolve temporary glitches.  
   Did this solve your issue? (yes/no)

--------------------------------------------------------------------------------
Replying as User. Provide feedback to IT_Support_Bot. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: Yes
User (to IT_Support_Bot):

Yes

--------------------------------------------------------------------------------
IT_Support_Bot (to User):

Great to hear that! If you encounter any further issues with Excel or have any other questions, feel free to

ChatResult(chat_id=59778814865590620028266872168506494443, chat_history=[{'content': 'Excel is not working', 'role': 'assistant', 'name': 'User'}, {'content': "I'm here to help you troubleshoot the issue with Excel. Let's go through some steps to identify and resolve the problem.\n\n1. **Restart Excel**: Close Excel completely and then reopen it. Sometimes, a simple restart can resolve temporary glitches.  \n   Did this solve your issue? (yes/no)", 'role': 'user', 'name': 'IT_Support_Bot'}, {'content': 'Yes', 'role': 'assistant', 'name': 'User'}, {'content': 'Great to hear that! If you encounter any further issues with Excel or have any other questions, feel free to ask. Have a wonderful day!', 'role': 'user', 'name': 'IT_Support_Bot'}, {'content': 'Thanks', 'role': 'assistant', 'name': 'User'}, {'content': "You're welcome! If you need assistance in the future, don't hesitate to reach out. Have a great day!", 'role': 'user', 'name': 'IT_Support_Bot'}], summary="You're welcome! If you n

## 5. A More Explicit Troubleshooting Conversation

This version makes the stopping behaviour clearer in the system message:

- If the user says **no**, provide another possible cause and next step.
- If the user says **yes**, provide a resolution message and stop.

`max_consecutive_auto_reply=10` acts as a safety limit. It does not guarantee that the problem will be solved in ten turns; it only prevents uncontrolled automatic continuation.

### Key teaching point

The agent's workflow is described mainly through natural-language instructions in `system_message`. AutoGen manages the conversation, while the LLM decides how to respond based on those instructions.


In [5]:
# ==========================================
# Setup API Key
# ==========================================
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API Key: ")

# ==========================================
# Imports
# ==========================================
import autogen

# ==========================================
# LLM Config for OpenRouter
# ==========================================
llm_config = {
    "config_list": [
        {
            "model": "openai/gpt-4o-mini",
            "api_key": os.getenv("OPENROUTER_API_KEY"),
            "base_url": "https://openrouter.ai/api/v1",
            "price": [0, 0],
        }
    ],
    "temperature": 0.4,
}

# ==========================================
# IT Support Agent
# ==========================================
it_support_agent = autogen.AssistantAgent(
    name="IT_Support_Bot",
    llm_config=llm_config,
    system_message="""
You are an IT support assistant.
Provide structured, step-by-step troubleshooting.
After each step, ask:
"Did this step solve your issue? (yes/no)"
If user says 'no', provide the next possible cause and step.
If user says 'yes', respond with a resolution message and stop.
"""
)

# ==========================================
# User Agent (Human-in-the-loop)
# ==========================================
user_agent = autogen.UserProxyAgent(
    name="User",
    human_input_mode="ALWAYS"
)

# ==========================================
# Start Chat
# ==========================================
issue = input("Describe your IT issue: ")

user_agent.initiate_chat(
    it_support_agent,
    message=issue
)


Enter your OpenRouter API Key: ··········
Describe your IT issue: My control panel is not opening
User (to IT_Support_Bot):

My control panel is not opening

--------------------------------------------------------------------------------
IT_Support_Bot (to User):

Let's troubleshoot the issue with your control panel not opening. 

**Step 1: Restart Your Computer**
Sometimes, a simple restart can resolve issues with applications not opening. Please restart your computer and then try to open the control panel again.

Did this step solve your issue? (yes/no)

--------------------------------------------------------------------------------
Replying as User. Provide feedback to IT_Support_Bot. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: no
User (to IT_Support_Bot):

no

--------------------------------------------------------------------------------
IT_Support_Bot (to User):

**Step 2: Check for Windows Updates**
Your system may need updates that could f

ChatResult(chat_id=102684122708508455871434188958832327668, chat_history=[{'content': 'My control panel is not opening', 'role': 'assistant', 'name': 'User'}, {'content': "Let's troubleshoot the issue with your control panel not opening. \n\n**Step 1: Restart Your Computer**\nSometimes, a simple restart can resolve issues with applications not opening. Please restart your computer and then try to open the control panel again.\n\nDid this step solve your issue? (yes/no)", 'role': 'user', 'name': 'IT_Support_Bot'}, {'content': 'no', 'role': 'assistant', 'name': 'User'}, {'content': '**Step 2: Check for Windows Updates**\nYour system may need updates that could fix the issue. Go to Settings > Update & Security > Windows Update, and check for updates. Install any available updates, then try to open the control panel again.\n\nDid this step solve your issue? (yes/no)', 'role': 'user', 'name': 'IT_Support_Bot'}], summary='**Step 2: Check for Windows Updates**\nYour system may need updates th

## 6. Dynamically Configure Agent Behaviour

This example asks the user to choose:

- A response style: detailed, concise, formal, or casual
- A troubleshooting priority: basic or advanced

These choices are inserted into a dynamically created system message.

```python
system_message = f"""
...
Response style: {response_style}
Troubleshooting priority: {troubleshooting_priority}
...
"""
```

This shows that an agent's behaviour does not have to be fixed. Its instructions can be constructed at runtime using user preferences, application settings, or business rules.

### Important distinction

This is **prompt-based configuration**, not model training. The underlying model remains the same; only the instructions sent to it change.


In [6]:

# ==========================================
# Securely Set OpenRouter API Key
# ==========================================
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API Key: ")

# ==========================================
# Imports
# ==========================================
import autogen

# ==========================================
# LLM Config (OpenRouter)
# ==========================================
llm_config = {
    "config_list": [
        {
            "model": "openai/gpt-4o-mini",
            "api_key": os.getenv("OPENROUTER_API_KEY"),
            "base_url": "https://openrouter.ai/api/v1",
            "price": [0.001, 0.0002],
        }
    ],
    "temperature": 0.4,
}

# ==========================================
# Get Behavior Configuration from User
# ==========================================
print("💻 Adaptive IT Support Bot")
print("Type 'exit' anytime to quit.\n")

response_style = input("Select response style (detailed / concise / formal / casual): ").strip().lower()
troubleshooting_priority = input("Select troubleshooting priority (basic / advanced): ").strip().lower()

# ==========================================
# Dynamic System Message Based on Config
# ==========================================
system_message = f"""
You are an adaptive IT support assistant.

Behavior settings:
- Response style: {response_style}
- Troubleshooting priority: {troubleshooting_priority}

Rules:
1. If troubleshooting priority is 'basic', suggest simple fixes first.
2. If troubleshooting priority is 'advanced', start with deeper diagnostics.
3. If response style is 'concise', keep response under 3 sentences.
4. If response style is 'formal', maintain professional tone.
5. If response style is 'casual', use friendly conversational tone.
6. Always provide clear and actionable troubleshooting steps.
"""

# ==========================================
# Create Assistant Agent
# ==========================================
it_support_agent = autogen.AssistantAgent(
    name="Adaptive_IT_Support",
    llm_config=llm_config,
    system_message=system_message,
)

# ==========================================
# Create User Proxy Agent
# ==========================================
user_agent = autogen.UserProxyAgent(
    name="User",
    human_input_mode="ALWAYS"
)

# ==========================================
# Start Chat Loop
# ==========================================
user_agent.initiate_chat(
    it_support_agent,
    message="Please describe your IT issue."
)


Enter your OpenRouter API Key: ··········
💻 Adaptive IT Support Bot
Type 'exit' anytime to quit.

Select response style (detailed / concise / formal / casual): detailed
Select troubleshooting priority (basic / advanced): advanced
User (to Adaptive_IT_Support):

Please describe your IT issue.

--------------------------------------------------------------------------------
Adaptive_IT_Support (to User):

I'm here to assist you with any IT-related issues you may be experiencing. Please provide a detailed description of the problem, including any error messages, specific software or hardware involved, and the steps you've already taken to resolve it. This will help me provide you with the most effective troubleshooting steps.

--------------------------------------------------------------------------------
Replying as User. Provide feedback to Adaptive_IT_Support. Press enter to skip and use auto-reply, or type 'exit' to end the conversation: Excel is crashing
User (to Adaptive_IT_Support

ChatResult(chat_id=114911490466695112462606197265000607516, chat_history=[{'content': 'Please describe your IT issue.', 'role': 'assistant', 'name': 'User'}, {'content': "I'm here to assist you with any IT-related issues you may be experiencing. Please provide a detailed description of the problem, including any error messages, specific software or hardware involved, and the steps you've already taken to resolve it. This will help me provide you with the most effective troubleshooting steps.", 'role': 'user', 'name': 'Adaptive_IT_Support'}, {'content': 'Excel is crashing', 'role': 'assistant', 'name': 'User'}, {'content': 'To address the issue of Excel crashing, let’s perform some advanced diagnostics:\n\n1. **Check for Updates**: Ensure that your Microsoft Office is up to date. Go to File > Account > Update Options > Update Now. This can resolve known bugs that may cause crashes.\n\n2. **Disable Add-ins**: Sometimes, third-party add-ins can cause Excel to crash. Start Excel in Safe Mo

# Part 3: Multi-Agent Collaboration

A multi-agent system assigns different responsibilities to specialised agents.

This example creates:

- `Diagnostic_Agent` — classifies the issue
- `Resolution_Agent` — proposes troubleshooting steps
- `Escalation_Agent` — prepares a critical-issue escalation
- `User` — represents the human
- `GroupChatManager` — manages the shared conversation

## GroupChat and GroupChatManager

`GroupChat` stores:

- The participating agents
- The shared message history
- The maximum number of conversation rounds

`GroupChatManager` coordinates the group chat and selects or manages the next interaction according to AutoGen's group-chat process.

```text
                  ┌─────────────────┐
                  │ Diagnostic Agent│
                  └────────┬────────┘
                           │
User ──→ Group Chat Manager├──→ Resolution Agent
                           │
                           └──→ Escalation Agent
```

Each agent has a specialised system message. This is the central idea behind role-based multi-agent design.


In [7]:

# ==========================================
# Securely Set OpenRouter API Key
# ==========================================
import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API Key: ")

# ==========================================
# Imports
# ==========================================
import autogen

# ==========================================
# LLM Config (OpenRouter)
# ==========================================
llm_config = {
    "config_list": [
        {
            "model": "openai/gpt-4o-mini",
            "api_key": os.getenv("OPENROUTER_API_KEY"),
            "base_url": "https://openrouter.ai/api/v1",
            "price": [0, 0],
        }
    ],
    "temperature": 0.4,
}




# ==========================================
# Diagnostic Agent
# ==========================================
diagnostic_agent = autogen.AssistantAgent(
    name="Diagnostic_Agent",
    llm_config=llm_config,
    system_message="""
You are a diagnostic IT agent.
Classify the issue into one of three categories:
- simple
- complex
- critical

Reply ONLY with one word: simple, complex, or critical.
"""
)

# ==========================================
# Resolution Agent
# ==========================================
resolution_agent = autogen.AssistantAgent(
    name="Resolution_Agent",
    llm_config=llm_config,
    system_message="""
You are an IT support specialist.
Provide structured step-by-step troubleshooting solutions.
"""
)

# ==========================================
# Escalation Agent
# ==========================================
escalation_agent = autogen.AssistantAgent(
    name="Escalation_Agent",
    llm_config=llm_config,
    system_message="""
You are an escalation agent.
If issue is critical, generate a professional escalation message
informing the user that senior IT intervention is required.
"""
)

# ==========================================
# User Agent
# ==========================================
user_agent = autogen.UserProxyAgent(
    name="User",
    human_input_mode="ALWAYS"
)

# ==========================================
# Group Chat Setup
# ==========================================
groupchat = autogen.GroupChat(
    agents=[user_agent, diagnostic_agent, resolution_agent, escalation_agent],
    max_round=10
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

# ==========================================
# Start Multi-Agent Workflow
# ==========================================
print("🤖 Multi-Agent IT Support System")
print("Type 'exit' to quit.\n")
issue = input("Describe your IT issue: ")

user_agent.initiate_chat(
    manager,
    message=issue
)


Enter your OpenRouter API Key: ··········
🤖 Multi-Agent IT Support System
Type 'exit' to quit.

Describe your IT issue: Excel not working
User (to chat_manager):

Excel not working

--------------------------------------------------------------------------------

Next speaker: Diagnostic_Agent

Diagnostic_Agent (to chat_manager):

simple

--------------------------------------------------------------------------------

Next speaker: Resolution_Agent

Resolution_Agent (to chat_manager):

To troubleshoot the issue with Excel not working, follow these structured steps:

### Step 1: Identify the Problem
1. **Clarify the Issue**: Ask the user what specific problem they are experiencing with Excel (e.g., won't open, crashes, error messages).
2. **Check for Error Messages**: If there are any error messages, note them down for further analysis.

### Step 2: Basic Checks
1. **Restart the Computer**: Sometimes, a simple restart can resolve many issues.
2. **Check for Updates**: Ensure that both 

ChatResult(chat_id=305468897826792337200662121293457376167, chat_history=[{'content': 'Excel not working', 'role': 'assistant', 'name': 'User'}, {'content': 'simple', 'name': 'Diagnostic_Agent', 'role': 'user'}, {'content': "To troubleshoot the issue with Excel not working, follow these structured steps:\n\n### Step 1: Identify the Problem\n1. **Clarify the Issue**: Ask the user what specific problem they are experiencing with Excel (e.g., won't open, crashes, error messages).\n2. **Check for Error Messages**: If there are any error messages, note them down for further analysis.\n\n### Step 2: Basic Checks\n1. **Restart the Computer**: Sometimes, a simple restart can resolve many issues.\n2. **Check for Updates**: Ensure that both Windows and Office are up to date.\n   - Go to **File** > **Account** > **Update Options** > **Update Now** in Excel.\n\n### Step 3: Safe Mode\n1. **Start Excel in Safe Mode**: This can help determine if the issue is caused by add-ins or custom settings.\n   

# Part 4: AutoGen and LangGraph Integration

AutoGen and LangGraph solve related but different problems.

- **AutoGen** provides conversational agents and multi-agent communication.
- **LangGraph** provides an explicit graph-based workflow with state, nodes, edges, and routing.

In the following example, AutoGen performs language understanding and response generation, while LangGraph controls which processing path should run.


## 7. Install LangGraph

LangGraph is installed separately because it is used to build the controlled retail-support workflow in the next section.


In [8]:
!pip install -U langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.9
    Uninstalling langgraph-1.2.9:
      Successfully uninstalled langgraph-1.2.9


## 8. Retail Support Workflow with State, Nodes, and Conditional Edges

### Shared state

`QueryState` is a `TypedDict` that defines the information passed through the graph:

- `user_input` — the original customer query
- `response` — the response produced by the workflow
- `category` — the classified query type

Think of state as a shared container that travels from one node to the next.

### Nodes

A node is a Python function that performs one step of work.

This graph contains:

- `FAQ_Classifier` — uses the AutoGen assistant to classify the query
- `Order_Handler` — looks up a simulated order status
- `Escalation` — returns a human-escalation response

### Edges

Edges define movement between nodes.

- A normal edge always moves to the same next node.
- A conditional edge chooses the next node using a routing function.

### Workflow

```text
START
  │
  ▼
FAQ_Classifier
  │
  ├── category = faq ───────→ END
  │
  ├── category = order ─────→ Order_Handler ─→ END
  │
  └── category = escalate ──→ Escalation ─────→ END
```

### Why combine the frameworks?

The LLM is flexible enough to understand the user's intent, while LangGraph provides predictable routing after classification. This gives the application both intelligence and workflow control.


In [1]:
# ==========================================================
# Install (Run Once in Google Colab)
# ==========================================================

# !pip install -U pyautogen langgraph openai


# ==========================================================
# Imports
# ==========================================================

import os
import re
from typing import TypedDict
from getpass import getpass

from autogen import AssistantAgent
from langgraph.graph import StateGraph, START, END


# ==========================================================
# Load OpenRouter API Key
# ==========================================================

os.environ["OPENROUTER_API_KEY"] = getpass(
    "Enter your OpenRouter API Key: "
)


# ==========================================================
# Define Shared Workflow State
# ==========================================================

class QueryState(TypedDict):
    user_input: str
    response: str
    category: str


# ==========================================================
# LLM Configuration — OpenRouter
# ==========================================================

llm_config = {
    "config_list": [
        {
            "base_url": "https://openrouter.ai/api/v1",
            "api_key": os.environ["OPENROUTER_API_KEY"],
            "model": "openai/gpt-4o-mini",

            # Prevent AutoGen price calculation warnings
            "price": [0, 0],
        }
    ],
    "temperature": 0.2,
}


# ==========================================================
# AutoGen Assistant Agent
# ==========================================================

assistant = AssistantAgent(
    name="Retail_Support_Bot",
    llm_config=llm_config,
    system_message="""
You are a professional retail customer-support assistant.

Classify each user query into exactly one of these categories:

1. faq
   General questions about products, delivery, returns,
   refunds, payments, or store policies.

2. order
   Questions about order tracking or order status.
   Any message containing a 5-digit order ID must be
   classified as order.

3. escalate
   Serious complaints or problems that require a human agent,
   such as fraud, legal complaints, repeated failed deliveries,
   or requests to speak to a manager.

Important rule:
A message containing only a 5-digit number is an order query.

Respond in exactly this format:

CATEGORY: faq
RESPONSE: Your response here

Replace faq with order or escalate when appropriate.
Do not add any other text.
""",
)


# ==========================================================
# Simulated Order Database/API
# ==========================================================

def get_order_status(order_id: str) -> str:
    """Return the status of an order from a simulated database."""

    fake_database = {
        "12345": "Shipped - Expected delivery in 2 days.",
        "67890": "Processing - Preparing for shipment.",
        "11111": "Delivered - Package delivered yesterday.",
    }

    return fake_database.get(
        order_id,
        "Order not found.",
    )


# ==========================================================
# Helper Function: Convert AutoGen Reply to Text
# ==========================================================

def extract_reply_text(reply) -> str:
    """
    AutoGen may return either a string or a dictionary.
    This function safely extracts the text.
    """

    if isinstance(reply, dict):
        return str(reply.get("content", ""))

    return str(reply)


# ==========================================================
# FAQ and Classification Node
# ==========================================================

def faq_classifier(state: QueryState):
    """
    Classify the query.

    A deterministic Python rule is used first for order IDs.
    Other queries are classified using the AutoGen agent.
    """

    user_input = state["user_input"].strip()

    # ------------------------------------------------------
    # Deterministic order-ID check
    # ------------------------------------------------------

    order_id_match = re.search(r"\b\d{5}\b", user_input)

    if order_id_match:
        return {
            "category": "order",
            "response": "",
        }

    # ------------------------------------------------------
    # Use the AutoGen agent for other queries
    # ------------------------------------------------------

    reply = assistant.generate_reply(
        messages=[
            {
                "role": "user",
                "content": user_input,
            }
        ]
    )

    reply_text = extract_reply_text(reply)

    # Uncomment this line if you want to inspect classification
    # print("Raw classifier output:", reply_text)

    # Default fallback values
    category = "faq"
    response_text = reply_text

    # ------------------------------------------------------
    # Parse the structured LLM response
    # ------------------------------------------------------

    if "CATEGORY:" in reply_text and "RESPONSE:" in reply_text:
        category_part, response_part = reply_text.split(
            "RESPONSE:",
            maxsplit=1,
        )

        category = (
            category_part
            .replace("CATEGORY:", "")
            .strip()
            .lower()
        )

        response_text = response_part.strip()

    # ------------------------------------------------------
    # Validate the generated category
    # ------------------------------------------------------

    allowed_categories = {"faq", "order", "escalate"}

    if category not in allowed_categories:
        category = "faq"

    return {
        "category": category,
        "response": response_text,
    }


# ==========================================================
# Order Handler Node
# ==========================================================

def order_handler(state: QueryState):
    """Extract the order ID and return its status."""

    match = re.search(
        r"\b\d{5}\b",
        state["user_input"],
    )

    if match:
        order_id = match.group()
        status = get_order_status(order_id)

        response = (
            f"Order #{order_id} Status:\n"
            f"{status}"
        )

    else:
        response = (
            "Please provide a valid 5-digit order ID.\n"
            "Example: 12345"
        )

    return {
        "response": response,
    }


# ==========================================================
# Human Escalation Node
# ==========================================================

def escalate_to_human(state: QueryState):
    """Return a message for queries requiring human assistance."""

    return {
        "response": (
            "Your issue requires human assistance.\n\n"
            "A senior support representative has been notified.\n"
            "Please wait while we connect you."
        )
    }


# ==========================================================
# Conditional Routing Function
# ==========================================================

def route_query(state: QueryState):
    """
    Select the next LangGraph node using the category
    produced by the classifier.
    """

    if state["category"] == "order":
        return "Order_Handler"

    if state["category"] == "escalate":
        return "Escalation"

    # FAQ response is already generated by the classifier
    return END


# ==========================================================
# Build LangGraph Workflow
# ==========================================================

graph = StateGraph(QueryState)

# Add nodes
graph.add_node(
    "FAQ_Classifier",
    faq_classifier,
)

graph.add_node(
    "Order_Handler",
    order_handler,
)

graph.add_node(
    "Escalation",
    escalate_to_human,
)

# Starting point
graph.add_edge(
    START,
    "FAQ_Classifier",
)

# Conditional routing
graph.add_conditional_edges(
    "FAQ_Classifier",
    route_query,
    {
        "Order_Handler": "Order_Handler",
        "Escalation": "Escalation",
        END: END,
    },
)

# End workflow after these nodes
graph.add_edge(
    "Order_Handler",
    END,
)

graph.add_edge(
    "Escalation",
    END,
)

# Compile the graph
workflow = graph.compile()


# ==========================================================
# CLI Interface
# ==========================================================

print("\nRetail Customer Support AI")
print("Powered by AutoGen + LangGraph\n")


while True:
    user_query = input(
        "Enter your query (or type 'exit'): "
    ).strip()

    if user_query.lower() == "exit":
        print("\nGoodbye!")
        break

    if not user_query:
        print("\nPlease enter a query.\n")
        continue

    initial_state: QueryState = {
        "user_input": user_query,
        "response": "",
        "category": "",
    }



    try:
        result = workflow.invoke(initial_state)

        print("\n--- AI Response ---\n")
        print(result["response"])

        # Optional: display the selected category
        print(f"\nCategory: {result['category']}")

        print("\n" + "=" * 60 + "\n")

    except Exception as error:
        print("\nAn error occurred:")
        print(error)
        print("\n" + "=" * 60 + "\n")

Enter your OpenRouter API Key: ··········

Retail Customer Support AI
Powered by AutoGen + LangGraph

Enter your query (or type 'exit'): order 91120

--- AI Response ---

Order #91120 Status:
Order not found.

Category: order


Enter your query (or type 'exit'): 11111

--- AI Response ---

Order #11111 Status:
Delivered - Package delivered yesterday.

Category: order


Enter your query (or type 'exit'): not got this order 

--- AI Response ---

Please provide a valid 5-digit order ID.
Example: 12345

Category: order


Enter your query (or type 'exit'): what is your company name?

--- AI Response ---

Our company name is [Company Name].

Category: faq


Enter your query (or type 'exit'): exit

Goodbye!


# Part 5: Tool-Enabled AutoGen Agent

An LLM does not automatically know live weather. To obtain current information, the application must call an external tool or API.

This section defines `get_weather(city)`, a normal Python function that queries SerpApi.

## Tool roles

- `weather_agent` decides that weather data is needed and formulates a response.
- `user_proxy` has access to the Python function through `function_map`.
- `get_weather` performs the actual external API request.

```text
User question
     ↓
Weather agent identifies the task
     ↓
Python weather function is executed
     ↓
Tool result is returned
     ↓
Agent presents the information
```

### Important classroom note

A function existing in Python is not enough by itself. It must be registered with the agent system so that the framework knows which function can be executed.


In [3]:
import os
import requests
from getpass import getpass
import autogen


os.environ["OPENROUTER_API_KEY"] = getpass(
    "Enter your OpenRouter API Key: "
)

os.environ["SERPAPI_KEY"] = getpass(
    "Enter your SerpApi Key: "
)


def get_weather_using_serp(city: str) -> str:
    """Fetch current weather for a city using SerpApi."""

    params = {
        "engine": "google",
        "q": f"weather in {city}",
        "api_key": os.environ["SERPAPI_KEY"],
    }

    response = requests.get(
        "https://serpapi.com/search",
        params=params,
        timeout=20,
    )

    if response.status_code != 200:
        return "Error fetching weather data from SerpApi."

    data = response.json()
    weather = data.get("answer_box")

    if not weather:
        return "Weather data not found. Please check the city name."

    temperature = weather.get("temperature", "N/A")
    condition = weather.get("weather", "N/A")
    humidity = weather.get("humidity", "N/A")
    wind = weather.get("wind", "N/A")

    return (
        f"Current weather in {city}:\n"
        f"Temperature: {temperature}\n"
        f"Condition: {condition}\n"
        f"Humidity: {humidity}\n"
        f"Wind: {wind}"
    )


llm_config = {
    "config_list": [
        {
            "model": "openai/gpt-4o-mini",
            "api_key": os.environ["OPENROUTER_API_KEY"],
            "base_url": "https://openrouter.ai/api/v1",
            "price": [0, 0],
        }
    ],
    "temperature": 0.2,
}


weather_agent = autogen.AssistantAgent(
    name="Weather_Agent",
    llm_config=llm_config,
    system_message=(
        "You are a helpful weather assistant. "
        "Use the get_weather_using_serp tool for current weather questions."
    ),
)

# Because Python while loop is already collecting the human’s input below:
user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=1,
)


weather_agent.register_for_llm(
    name="get_weather_using_serp",
    description="Get the current weather for a city.")(get_weather_using_serp)

user_proxy.register_for_execution(name="get_weather_using_serp")(get_weather_using_serp)




while True:
    query = input("Ask about weather, or type 'exit': ")

    if query.lower() == "exit":
        print("Goodbye")
        break

    user_proxy.initiate_chat(
        weather_agent,
        message=query,
        clear_history=True,
    )

Enter your OpenRouter API Key: ··········
Enter your SerpApi Key: ··········
Ask about weather, or type 'exit': Berlin
User (to Weather_Agent):

Berlin

--------------------------------------------------------------------------------
Weather_Agent (to User):

***** Suggested tool call (call_wB5IBZJrugfEa2CIRdmJukCR): get_weather_using_serp *****
Arguments: 
{"city":"Berlin"}
***************************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION get_weather_using_serp...
Call ID: call_wB5IBZJrugfEa2CIRdmJukCR
Input arguments: {'city': 'Berlin'}

>>>>>>>> EXECUTED FUNCTION get_weather_using_serp...
Call ID: call_wB5IBZJrugfEa2CIRdmJukCR
Input arguments: {'city': 'Berlin'}
Output:
Current weather in Berlin:
Temperature: 74
Condition: Partly sunny
Humidity: 59%
Wind: 5 mph
User (to Weather_Agent):

***** Response from calling tool (call_wB5IBZJrugfEa2CIRdmJukCR) *

KeyboardInterrupt: Interrupted by user

# Part 6: OCR and Multi-Agent Contract Analysis

The final example combines document processing with multiple specialised agents.

The installation cell adds:

- `pytesseract` for optical character recognition
- `pdf2image` for converting PDF pages to images
- `Pillow` for image handling
- Poppler utilities for PDF conversion
- The Tesseract OCR engine

> OCR is useful for scanned PDFs. Its accuracy depends on scan quality, page layout, language, and image clarity.


In [4]:

!pip install pyautogen pytesseract pdf2image pillow

!apt-get install -y tesseract-ocr

!apt-get update && apt-get install -y poppler-utils


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 24 not upgraded.
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [107 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.7 MB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu j

## 9. Document Analysis Pipeline

This example has two broad stages.

### Stage 1: Extract text with OCR

```text
Uploaded PDF
     ↓
Convert every page into an image
     ↓
Run Tesseract OCR on each image
     ↓
Combine extracted text
```

The extracted text is stored in `contract_text`.

### Stage 2: Analyse with specialised agents

Four assistant agents review the same contract from different perspectives:

1. `Document_Ingestor` summarises important clauses.
2. `Compliance_Checker` looks for compliance concerns.
3. `Risk_Assessor` identifies legal and financial risks.
4. `Revision_Recommender` proposes improvements.

A `GroupChatManager` coordinates their shared conversation.

```text
Contract text
     ↓
Document Ingestor
     ↓
Compliance Checker
     ↓
Risk Assessor
     ↓
Revision Recommender
     ↓
Combined multi-agent analysis
```

### Why use multiple agents?

Separating roles can make a complex task easier to organise and inspect. Each agent has a focused responsibility instead of asking one prompt to perform every type of analysis at once.

### Limitation

The code requests a preferred workflow in the initial message, but the group-chat manager still controls the actual speaker sequence. A stricter deterministic order would require additional orchestration logic.


In [5]:
# ============================================================
# Imports
# ============================================================
import pytesseract
from pdf2image import convert_from_path
import autogen
from google.colab import files
from getpass import getpass

# ============================================================
# Get API Key
# ============================================================
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API Key: ")

# Configure Tesseract path
pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"

# ============================================================
# Upload PDF
# ============================================================
print("\n Upload your contract PDF")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

# ============================================================
# OCR Extraction
# ============================================================
def extract_text_from_pdf(pdf_path: str) -> str:
    images = convert_from_path(pdf_path)
    text = "\n".join(
        pytesseract.image_to_string(img, config="--psm 6")
        for img in images
    )
    return text

print("\n🔍 Extracting text...")
contract_text = extract_text_from_pdf(pdf_filename)

if not contract_text.strip():
    print("❌ No text extracted.")
    exit()

print("\n✅ Text extraction complete.\n")


Enter your OpenRouter API Key: ··········

 Upload your contract PDF


Saving Ocean's Surprise - Pitch Deck.pdf to Ocean's Surprise - Pitch Deck.pdf

🔍 Extracting text...

✅ Text extraction complete.



In [6]:

# ============================================================
# LLM Config (OpenRouter)
# ============================================================
llm_config = {
    "config_list": [
        {
            "model": "openai/gpt-4o-mini",
            "api_key": OPENROUTER_API_KEY,
            "base_url": "https://openrouter.ai/api/v1",
        }
    ],
    "temperature": 0.3,
}

# ============================================================
# Define Agents
# ============================================================

doc_ingestor = autogen.AssistantAgent(
    name="Document_Ingestor",
    system_message="""
Extract and summarize key clauses from the contract.
Focus on payment terms, liability, termination, confidentiality, and obligations.
""",
    llm_config=llm_config,
)

compliance_checker = autogen.AssistantAgent(
    name="Compliance_Checker",
    system_message="""
Review the contract for GDPR, corporate compliance, and regulatory concerns.
Refer to Document_Ingestor summary when possible.
""",
    llm_config=llm_config,
)

risk_assessor = autogen.AssistantAgent(
    name="Risk_Assessor",
    system_message="""
Analyze legal and financial risks.
Identify missing clauses or ambiguous language.
Refer to previous agents' findings.
""",
    llm_config=llm_config,
)

revision_recommender = autogen.AssistantAgent(
    name="Revision_Recommender",
    system_message="""
Suggest improvements to reduce risks and improve compliance.
Use insights from all previous agents.
""",
    llm_config=llm_config,
)

user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    code_execution_config=False,
)

groupchat = autogen.GroupChat(
    agents=[
        user_proxy,
        doc_ingestor,
        compliance_checker,
        risk_assessor,
        revision_recommender,
    ],
    messages=[],
    max_round=5,
    speaker_selection_method="round_robin",
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

# ============================================================
# Start Multi-Agent Collaboration
# ============================================================

print("\n Running Collaborative Contract Analysis...\n")

user_proxy.initiate_chat(
    manager,
    message=f"""
Analyze the following contract collaboratively.

Workflow:
1. Document_Ingestor summarizes key clauses.
2. Compliance_Checker reviews compliance.
3. Risk_Assessor identifies risks.
4. Revision_Recommender suggests improvements.

Contract:
{contract_text}
"""
)

print("\n✅ Analysis Completed.\n")



 Running Collaborative Contract Analysis...

User (to chat_manager):


Analyze the following contract collaboratively.

Workflow:
1. Document_Ingestor summarizes key clauses.
2. Compliance_Checker reviews compliance.
3. Risk_Assessor identifies risks.
4. Revision_Recommender suggests improvements.

Contract:
ns : <a rr. - , 3 >
- - . a al . 4 %
_ eS — ‘ F etre y 4, 7 " -
| ii a \ a
99 7 oe om,
_¢ ’ : is P >
4 : £5 =
Seed stage “a 9 = By
. : ’ dl ’ ee ae j
Investment Pitch 7 Se |
— aed e ~~
Building India’s only Seafood, Meat & Poultry Supply Chain ; =e “ a ee
for HORECA & Supermarkets with fully managed Farms, hide" “uP Pas ae
Integrated Processing Unit, Cold Storage and Distribution - > -& aN
ee ve es we - at - x ot m . .
Safe ae E \ ) ae
tsar x . : » * ? me ae A
| At Yr. ; IS aa 3?
’ et PF m a af. , ; as ™“
ee ’ 4 a ia te ee - ™~
i A “A Paes, Ny
ee ge
© Silver Blade Foods Pvt. Ltd. 2026 Draft and Confidentiag a b j r. el C oN rae k , w-

ia’s Meat & Fish lies: Where does i f eC.
In

Document_Ingestor (to chat_manager):

### Key Clauses Summary

#### Payment Terms
- **Investment Ask**: INR 5 Crore for 20% equity.
- **Projected Revenue**: Anticipated growth to INR 300 Lakhs in ARR for FY 2026-27.
- **Sales Invoices**: Total sales amount projected at INR 46.80 Lakhs for the period of April to June 2026, with a net margin of approximately 18%.

#### Liability
- **Liability Limitations**: The contract does not explicitly mention liability limitations; however, the founders emphasize the importance of compliance with food safety norms and regulations to mitigate risks associated with product quality and safety.

#### Termination
- **Termination Conditions**: The contract does not specify termination clauses, indicating a need for clarity on conditions under which either party can terminate the agreement.

#### Confidentiality
- **Confidentiality Clause**: The document is marked as confidential, suggesting that sensitive business information should not be disclosed witho

Compliance_Checker (to chat_manager):

### Compliance_Checker Review

#### GDPR Compliance
- **Data Handling**: The contract does not explicitly mention data handling practices in relation to GDPR. If personal data of customers or suppliers is collected, processed, or stored, the company must ensure compliance with GDPR principles, including obtaining consent, ensuring data security, and providing rights to data subjects.
- **Privacy Policy**: There should be a clear privacy policy in place that outlines how personal data is managed, which is crucial for GDPR compliance.

#### Corporate Compliance
- **Regulatory Adherence**: The commitment to comply with food safety and hygiene standards is a positive aspect. However, it is essential to ensure that all operational practices align with local and national regulations governing the seafood and meat industry.
- **Licensing and Permits**: The company should verify that all necessary licenses and permits for processing, storage, and distribu

Risk_Assessor (to chat_manager):

### Collaborative Contract Analysis Summary

#### Document_Ingestor Summary
The contract outlines key clauses related to payment terms, liability, termination, confidentiality, and operational obligations. It emphasizes the company's commitment to building an integrated supply chain for seafood, meat, and poultry while adhering to food safety regulations.

#### Compliance_Checker Review
The compliance review highlights the need for GDPR compliance regarding data handling practices, as well as the importance of adhering to food safety and environmental regulations. It emphasizes the necessity of obtaining relevant licenses and permits for operations.

#### Risk_Assessor Review
The risk assessment identifies significant market competition, supply chain vulnerabilities, and potential financial challenges as key risks. Suggested mitigation strategies include diversifying suppliers, conducting regular market analysis, and developing a robust financial plan.

Revision_Recommender (to chat_manager):

### Comprehensive Improvements to Reduce Risks and Improve Compliance

Based on the collaborative analysis of the contract, here are detailed recommendations to enhance compliance, mitigate risks, and ensure operational clarity:

#### 1. **Incorporate GDPR Provisions**
   - **Data Protection Measures**: Include explicit clauses that outline how personal data will be collected, processed, stored, and shared. Specify the legal basis for data processing and the rights of data subjects (e.g., access, rectification, erasure).
   - **Privacy Policy**: Develop and attach a comprehensive privacy policy that details data handling practices, including how data breaches will be managed and reported.

#### 2. **Specify Regulatory Compliance**
   - **Food Safety Regulations**: Clearly reference compliance with the Food Safety and Standards Authority of India (FSSAI) guidelines and any other relevant local or national food safety regulations. This should incl

# Summary: What Progressed from Basic to Advanced?

| Stage | Main concept |
|---|---|
| Basic support chat | Two agents exchanging messages |
| IT troubleshooting | Human-in-the-loop and conversational context |
| Adaptive support | Dynamic system messages |
| Multi-agent helpdesk | Specialised roles and group chat |
| AutoGen + LangGraph | Explicit state and conditional routing |
| Weather assistant | External tool/API integration |
| Contract analysis | OCR plus collaborative specialised agents |

## Core terms to remember

- **Agent:** A participant with a name, role, behaviour, and optional LLM/tool access
- **AssistantAgent:** Usually an LLM-powered agent
- **UserProxyAgent:** Represents a user or execution proxy
- **System message:** Instructions defining an agent's role and rules
- **GroupChat:** Shared conversation among multiple agents
- **GroupChatManager:** Coordinates the group conversation
- **Tool:** A Python function or API used to obtain or perform something outside the LLM
- **State:** Shared workflow data in LangGraph
- **Node:** One processing step in a graph
- **Edge:** A connection that controls where execution moves next
- **Conditional edge:** A route selected according to state or logic
